# Part 1: Pothole dataset

Run the cells **from top to bottom** in Google Colab.

## 1. Understand the dataset

- **Source:** [Pothole v1 on Roboflow](https://universe.roboflow.com/brad-dwyer/pothole-voxrl/dataset/1), by Atikur Rahman Chitholian.
- **Usage terms:** [ODbL v1.0](https://opendatacommons.org/licenses/odbl/1-0/), as listed by Roboflow. Retain attribution and applicable license notices when sharing the dataset.
- **Why this dataset:** 665 images, already annotated, and one class make the workload manageable.
- **Task:** find potholes and draw boxes around them. Planned small detector: **YOLOv8n**.
- **Class:** `0 = pothole`. No potholes means no box annotations, not a second class.
- **YOLO TXT format:** one row per pothole: `class_id x_center y_center width height`. Box values are normalized to 0–1.
- **Existing split:** 465 train, 133 validation, 67 test images. We use these provided folders; we do not create a new split or choose a splitting seed.

**For the defense:** training data teaches the model, validation compares the runs, and test data evaluates the selected model.

## 2. Load the dataset

Download the dataset as a **YOLOv8 ZIP**. This cell reuses `/content/pothole` if it is already extracted. Otherwise, choose the ZIP when the file picker opens.

`root` is the dataset folder. `data_yaml` tells YOLO where the images are.

In [ ]:
import subprocess
import sys

# Pillow opens images and draws boxes. The other imports come with Python/Colab.
_ = subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow==11.0.0"])


In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display, Markdown
import zipfile
import hashlib

folder = Path("/content/pothole")
yaml_files = list(folder.rglob("data.yaml"))

if not yaml_files:
    from google.colab import files
    uploaded = files.upload()  # Select the YOLOv8 dataset ZIP.
    zip_name = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(folder)
    yaml_files = list(folder.rglob("data.yaml"))

assert len(yaml_files) == 1, "Use one YOLOv8 dataset export containing data.yaml."
root = yaml_files[0].parent.resolve()
data_yaml = root / "data.yaml"

# Set explicit paths for the later training cells.
_ = data_yaml.write_text(
    f"path: {root.as_posix()}\ntrain: train/images\nval: valid/images\n"
    "test: test/images\nnames:\n  0: pothole\n"
)
print("Dataset folder:", root)


## 3. Count images and check labels

One image can contain several potholes. Count **valid boxes** separately from images and report invalid rows instead of silently removing them.

Missing or empty labels need visual review: they are valid only when the image has no potholes.

In [ ]:
def valid_label(line):
    try:
        class_id, x, y, w, h = map(float, line.split())
    except ValueError:
        return False
    return (class_id == 0 and w > 0 and h > 0
            and -0.00001 <= x-w/2 < x+w/2 <= 1.00001
            and -0.00001 <= y-h/2 < y+h/2 <= 1.00001)

image_files, issues = {}, []
table = "| Split | Images | Valid boxes |\n| --- | ---: | ---: |\n"

for split in ["train", "valid", "test"]:
    images = sorted(p for p in (root / split / "images").glob("*")
                    if p.suffix.lower() in [".jpg", ".jpeg", ".png"])
    assert images, f"No images found in {split}/images. Check the dataset folder."
    image_files[split] = images
    objects = 0

    for image in images:
        label = root / split / "labels" / f"{image.stem}.txt"
        if not label.exists():
            issues.append(f"Missing label: {split}/{image.name}")
            continue
        lines = [line for line in label.read_text().splitlines() if line.strip()]
        if not lines:
            issues.append(f"Empty label: {split}/{image.name}; review the image")
        for number, line in enumerate(label.read_text().splitlines(), 1):
            if line.strip():
                if valid_label(line):
                    objects += 1
                else:
                    issues.append(f"Invalid box: {split}/{label.name}, line {number}")

    table += f"| {split} | {len(images)} | {objects} |\n"

display(Markdown(table))
print("Label issues to review:", len(issues))
for issue in issues:
    print(issue)


## 4. Check duplicates across splits

An exact file fingerprint finds identical files. A small grayscale brightness pattern flags **possible** similar images; these matches can be false positives.

Review the listed pairs. Confirmed duplicates, near-duplicates and frames from the same video must stay in **one split**. This check cannot prove that every source-video relationship has been found.

In [ ]:
fingerprints = []
for split, images in image_files.items():
    for path in images:
        exact = hashlib.sha256(path.read_bytes()).hexdigest()
        with Image.open(path) as original:
            small = ImageOps.exif_transpose(original).convert("L").resize((8, 8))
            pixels = list(small.getdata())
        average = sum(pixels) / len(pixels)
        pattern = [pixel > average for pixel in pixels]
        fingerprints.append((split, path, exact, pattern))

matches = []
for i, first in enumerate(fingerprints):
    for second in fingerprints[:i]:
        if first[0] == second[0]:  # Copies inside the same split do not cause cross-split leakage.
            continue
        difference = sum(a != b for a, b in zip(first[3], second[3]))
        if first[2] == second[2]:
            matches.append((first[1], second[1], "Exact duplicate"))
        elif difference <= 5:  # A heuristic: review these pairs visually.
            matches.append((first[1], second[1], "Possible near-duplicate"))

print("Cross-split duplicate candidates:", len(matches))
for first, second, kind in matches:
    print(kind, first.relative_to(root), "and", second.relative_to(root))
if not matches:
    print("No matches detected by this check; review related scenes/video sources too.")


If candidates were found, this cell shows the first five pairs for review. Change `[0:5]` to `[5:10]` to inspect the next five, and continue until you have reviewed the listed candidates.

In [ ]:
for first, second, kind in matches[0:5]:
    print(kind)
    for path in [first, second]:
        print(path.relative_to(root))
        with Image.open(path) as original:
            preview = ImageOps.exif_transpose(original).convert("RGB")
        preview.thumbnail((400, 300))
        display(preview)


## 5. Display five labeled images

Green rectangles are **ground-truth boxes from the dataset**, not predictions. Check whether they cover the potholes and whether any visible potholes are missing labels.

In [ ]:
for path in image_files["train"][:5]:
    with Image.open(path) as original:
        image = ImageOps.exif_transpose(original).convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)
    label = root / "train/labels" / f"{path.stem}.txt"

    if label.exists():
        for line in label.read_text().splitlines():
            if not line.strip() or not valid_label(line):
                continue  # Invalid rows were reported in section 3; they still need attention.
            _, x, y, w, h = map(float, line.split())
            draw.rectangle([(x-w/2)*width, (y-h/2)*height,
                            (x+w/2)*width, (y+h/2)*height], outline="lime", width=3)

    image.thumbnail((800, 600))
    print(path.name)
    display(image)


## 6. Add the required information to README.md

Copy the dataset source, license, class, format and model choice from section 1, and the actual count table from section 3. Add a few sentences about the label issues and duplicate candidates you reviewed and how you resolved confirmed problems.

- Fix incorrect labels and confirmed cross-split related images **before training**. These cells report findings; they do not automatically delete or move files.
- If you keep the provider split, say so. If you create a new split, record the grouping method and random seed and use the same split for both runs.
- Source-video IDs are not reliably supplied. Review available provenance/related scenes and document any uncertainty; do not claim zero leakage based only on hashes.

**For the defense:** “We counted the images and potholes, checked the labels, reviewed duplicate candidates, and inspected five labeled images.” Use that statement only after doing the reviews.

Code assistance: OpenAI ChatGPT. Part 2 below trains the baseline detector.

# Part 2: Train the baseline detector

We fine-tune **YOLOv8n**: start with COCO-pretrained weights and adjust them using
our pothole training images. Validation results select the best checkpoint.

Before starting, finish Part 1, correct its invalid labels and resolve confirmed
cross-split duplicates/related frames. The test set is reserved for Part 4.

In Colab, choose **Runtime → Change runtime type → T4 GPU**. If changing the
runtime removes your files or variables, rerun Part 1 first.

## 2.1 Install the detector library

In [ ]:
import subprocess
import sys

_ = subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.228"
])

from ultralytics import YOLO
import ultralytics
import torch
import time
import json

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)


## 2.2 Set the baseline settings

Use the same dataset configuration created in Part 1. The seed makes the run more
reproducible. SGD is selected explicitly so the stated learning rate is used.

In [ ]:
assert "data_yaml" in globals() and "issues" in globals(), "Run Part 1 first."
assert data_yaml.exists(), "Dataset missing: reload it using Part 1."
assert not any(issue.startswith("Invalid box:") for issue in issues), \
    "Correct the invalid boxes from Part 1, then rerun its label-check cell."

INITIAL_WEIGHTS = "yolov8n.pt"
settings = {
    "data": str(data_yaml.resolve()),
    "imgsz": 416,           # Training image size.
    "epochs": 30,           # Full passes through the training images.
    "batch": 8,             # Images processed in one training batch.
    "optimizer": "SGD",
    "lr0": 0.01,            # Initial learning rate.
    "seed": 42,
    "patience": 0,          # Disable early stopping; run all 30 epochs.
    "device": 0 if torch.cuda.is_available() else "cpu",
    "workers": 2,
    "val": True, "save": True, "plots": True,
    "project": "runs/pothole",
    "name": "baseline_416"
}
print("Device:", settings["device"])


## 2.3 Train and record the run

The model learns from training images. Validation images measure performance
without updating the weights. Ultralytics saves the best validation checkpoint as
**best.pt**. Training time is measured around the training call.

In [ ]:
baseline = YOLO(INITIAL_WEIGHTS)  # Load the pretrained detector.

start = time.perf_counter()
baseline.train(**settings)
baseline_minutes = (time.perf_counter() - start) / 60

# Use the actual run folder; YOLO may add a number if the name already exists.
run_dir = Path(baseline.trainer.save_dir).resolve()
baseline_checkpoint = str(Path(baseline.trainer.best).resolve())
run_info = {
    "framework": "Ultralytics", "version": ultralytics.__version__,
    "pytorch_version": torch.__version__, "model": "YOLOv8n",
    "initial_weights": INITIAL_WEIGHTS, **settings,
    "training_minutes": baseline_minutes,
    "best_checkpoint": baseline_checkpoint
}
_ = (run_dir / "baseline_info.json").write_text(json.dumps(run_info, indent=2))
print(f"Training time: {baseline_minutes:.2f} minutes")
print("Best checkpoint:", baseline_checkpoint)


## 2.4 Show the curves and save the required outputs

| Saved file | What it contains |
| --- | --- |
| `args.yaml` | Full training configuration |
| `results.csv` | Losses and validation metrics for each epoch |
| `results.png` | Learning curves |
| `weights/best.pt` | Best checkpoint selected using validation results |
| `baseline_info.json` | Model, initial weights, settings, versions and elapsed time |

The ZIP below preserves this run's outputs before the Colab session ends.
Copy the settings and measured training time into the project README.

In [ ]:
display(Image.open(run_dir / "results.png"))

import shutil
from google.colab import files

archive = shutil.make_archive("baseline_results", "zip", root_dir=run_dir)
files.download(archive)


# Part 3: Run one controlled experiment

**Hypothesis, before training:** increasing image size from **416 to 640** may
help detect small potholes, but may take longer to train.

We use a fresh YOLOv8n from the **same initial pretrained weights**. Keep the data
split and the Part 2 settings unchanged. Only the training image-size setting
changes. A different run name only keeps the saved outputs in separate folders.

Run these cells after Part 2 in the **same Colab session**. If the baseline has
already trained, copy only the Part 3 cells into your current notebook; its
variables and checkpoint remain available there.

## 3.1 Train the experiment

In [ ]:
assert "baseline_checkpoint" in globals() and Path(baseline_checkpoint).exists(), \
    "Finish Part 2 first and keep this Colab session open."

# Copy the baseline settings; change only image size and the output-folder name.
experiment_settings = settings.copy()
experiment_settings.update(imgsz=640, name="experiment_640")
experiment = YOLO(INITIAL_WEIGHTS)  # Start from pretrained weights, not baseline weights.

start = time.perf_counter()
experiment.train(**experiment_settings)
experiment_minutes = (time.perf_counter() - start) / 60

experiment_dir = Path(experiment.trainer.save_dir).resolve()
experiment_checkpoint = str(Path(experiment.trainer.best).resolve())
experiment_info = {
    **run_info, **experiment_settings,
    "training_minutes": experiment_minutes,
    "best_checkpoint": experiment_checkpoint
}
print(f"Experiment training time: {experiment_minutes:.2f} minutes")


## 3.2 Compare the two runs on validation images

Evaluate both best checkpoints at **640 pixels**, using identical evaluation
settings and the same validation images. The table's image size refers to the
**training** size. We do not use test images in this comparison.

- **Precision:** how many reported potholes are correct.
- **Recall:** how many annotated potholes are found.
- **mAP@0.50:** detection quality across confidence thresholds, with matching IoU
  of at least 0.50.

The confidence floor is 0.001 and NMS IoU is 0.70. Ultralytics summarizes precision
and recall at its maximum-F1 operating point.

In [ ]:
import csv

evaluation_settings = {
    "data": str(data_yaml.resolve()), "split": "val",
    "imgsz": 640, "batch": 8, "device": settings["device"],
    "conf": 0.001, "iou": 0.7, "rect": False,
    "plots": False, "verbose": False,
    "project": "runs/pothole_validation"
}
runs = [
    ("Baseline", settings["imgsz"], baseline_checkpoint, baseline_minutes),
    ("Experiment", 640, experiment_checkpoint, experiment_minutes)
]
comparison = []
for name, size, checkpoint, minutes in runs:
    metrics = YOLO(checkpoint).val(**evaluation_settings, name=name.lower())
    comparison.append({
        "Run": name, "Training size": size,
        "Precision": float(metrics.box.mp), "Recall": float(metrics.box.mr),
        "mAP@0.50": float(metrics.box.map50), "Minutes": minutes
    })

comparison_file = experiment_dir / "comparison.csv"
with comparison_file.open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(comparison[0]))
    writer.writeheader()
    writer.writerows(comparison)

comparison_table = (
    "| Run | Training size | Precision | Recall | mAP@0.50 | Minutes |\n"
    "| --- | ---: | ---: | ---: | ---: | ---: |\n"
)
for row in comparison:
    comparison_table += (
        f"| {row['Run']} | {row['Training size']} | {row['Precision']:.4f} | "
        f"{row['Recall']:.4f} | {row['mAP@0.50']:.4f} | {row['Minutes']:.2f} |\n"
    )
display(Markdown(comparison_table))

map_change = comparison[1]["mAP@0.50"] - comparison[0]["mAP@0.50"]
time_change = experiment_minutes - baseline_minutes
print(f"mAP@0.50 change (640 minus 416): {map_change:+.4f}")
print(f"Training time change (minutes): {time_change:+.2f}")


### Explain your actual result

Use the table to write **two short sentences** in the README:

1. Did the larger image size improve validation mAP@0.50? Mention the measured
   difference and any precision/recall trade-off.
2. Was the change worth its measured training time? If it did not help, possible
   explanations include larger images adding little useful detail or the small
   dataset making results variable. Treat causes as hypotheses, not proven facts.

An improvement is **not required**. A fair comparison and a thoughtful explanation
are required. These are validation results, not final test results.

## 3.3 Save the experiment and comparison

The experiment ZIP contains its configuration, logs, curves, best checkpoint and
run information. It also includes the comparison CSV, dataset configuration and
the dependency versions explicitly installed by this notebook. The actual
PyTorch version is recorded in each run's information file.

In [ ]:
import shutil
from google.colab import files

display(Image.open(experiment_dir / "results.png"))

# Add the shared configuration and comparison to this run's download.
experiment_info["evaluation_settings"] = evaluation_settings
_ = (experiment_dir / "experiment_info.json").write_text(json.dumps(experiment_info, indent=2))
shutil.copy2(data_yaml, experiment_dir / "data.yaml")
_ = (experiment_dir / "requirements.txt").write_text("ultralytics==8.3.228\nPillow==11.0.0\n")

experiment_archive = shutil.make_archive("experiment_results", "zip", root_dir=experiment_dir)
files.download(experiment_archive)


## What should I download?

**Before running:** the YOLOv8 dataset ZIP and this notebook are enough. The code
installs the libraries and YOLO downloads `yolov8n.pt` automatically.

**After running Parts 1–3, keep:**

| File | How to get it |
| --- | --- |
| Executed notebook (`.ipynb`) | In Colab: File → Download → Download .ipynb. Save the copy with your actual outputs. |
| `baseline_results.zip` | Downloaded at the end of Part 2. |
| `experiment_results.zip` | Downloaded at the end of Part 3. |

Inside the ZIPs are `weights/best.pt`, `args.yaml`, `results.csv`, `results.png`
and the run information. The experiment ZIP also contains `comparison.csv`,
`data.yaml` and `requirements.txt`. **No separate manual model download is needed.**

Create README.md separately for GitHub. Copy your dataset information and actual
comparison table into it, and add the two-sentence explanation above.

This notebook currently covers Parts 1–3. Part 4 (held-out test evaluation and
failure analysis) and Part 5 (final repository preparation) remain.